In [8]:
"""
Nepali NLP Pipeline  —  Full Steps
=====================================
Step 0 : Normalize suffixes.json + stopwords.json  → save normalized versions
Step 1 : Tokenization          (whitespace + em-dash split)
Step 2 : Stopword removal      (using normalized stopwords.json)
Step 3 : Suffix stripping      (using normalized suffixes.json, longest-first)
Step 4 : Token frequency table (raw / after stopword removal / after stemming)
Step 5 : Vocabulary index      (word2id / id2word on stemmed tokens)
Step 6 : Verification report

Outputs
-------
  normalized_stopwords.json
  normalized_suffixes.json
  pipeline_dataset.csv          — all intermediate columns per sentence
  token_freq.csv                — freq at each stage
  vocab.json                    — word2id, id2word, stopwords set, suffix list
  pipeline_report.txt           — verification report
"""

import json, csv, re, collections, unicodedata
from pathlib import Path

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# PATHS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
NORM_MAP_PATH  = "normalization_map.json"
SUFFIXES_PATH  = "suffixes.json"
STOPWORDS_PATH = "stopwords.json"
DATASET_PATH   = "normalized_dataset.csv"

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# LOAD RAW FILES
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
with open(NORM_MAP_PATH,  encoding="utf-8") as f: norm_map  = json.load(f)
with open(SUFFIXES_PATH,  encoding="utf-8") as f: raw_sfx   = json.load(f)["suffix"]
with open(STOPWORDS_PATH, encoding="utf-8") as f: raw_sw    = json.load(f)["stopwords"]

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 0 — NORMALIZATION FUNCTION  (same as normalize.py)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
_sorted_keys      = sorted(norm_map.keys(), key=len, reverse=True)
_protected        = set("".join(norm_map.values()))   # chars used in replacements

def normalize(text: str) -> str:
    text = unicodedata.normalize("NFC", text)
    for key in _sorted_keys:
        rep = norm_map[key]
        # don't strip a char that appears inside a replacement value (e.g. ् in ग्य)
        if rep == "" and key in _protected:
            continue
        if key in text:
            text = text.replace(key, rep)
    return " ".join(text.split())

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 0 — NORMALIZE STOPWORDS & SUFFIXES  →  deduplicated sets
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
stopword_set: set[str] = set()
for w in raw_sw:
    norm = normalize(w)
    if norm:                        # skip if normalized to empty string
        stopword_set.add(norm)

# Suffixes: deduplicate then sort longest-first for greedy matching
suffix_list: list[str] = sorted(
    {normalize(s) for s in raw_sfx if normalize(s)},
    key=len, reverse=True
)

# Save normalized lists
with open("normalized_stopwords.json", "w", encoding="utf-8") as f:
    json.dump({"stopwords": sorted(stopword_set)}, f, ensure_ascii=False, indent=2)

with open( "normalized_suffixes.json", "w", encoding="utf-8") as f:
    json.dump({"suffixes": suffix_list}, f, ensure_ascii=False, indent=2)

print(f"Step 0  ✅  Stopwords : {len(raw_sw)} raw → {len(stopword_set)} normalized unique")
print(f"Step 0  ✅  Suffixes  : {len(raw_sfx)} raw → {len(suffix_list)} normalized unique")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 1 — TOKENIZER
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def tokenize(text: str) -> list[str]:
    """Whitespace tokenizer that also splits on em-dash / hyphen."""
    tokens = []
    for t in text.split():
        parts = re.split(r'[–\-]', t)
        tokens.extend(p for p in parts if p)
    return tokens

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 2 — STOPWORD REMOVAL
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def remove_stopwords(tokens: list[str]) -> list[str]:
    return [t for t in tokens if t not in stopword_set]

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 3 — SUFFIX STRIPPING  (greedy longest-first)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
MIN_ROOT_LEN = 2   # don't strip if root would be shorter than this

def strip_suffix(token: str) -> tuple[str, str | None]:
    """Return (root, matched_suffix) or (token, None) if nothing stripped."""
    for sfx in suffix_list:
        if token.endswith(sfx) and len(token) - len(sfx) >= MIN_ROOT_LEN:
            return token[: -len(sfx)], sfx
    return token, None

def stem_tokens(tokens: list[str]) -> list[str]:
    return [strip_suffix(t)[0] for t in tokens]

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# RUN PIPELINE ON DATASET
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
rows_in: list[dict] = []
with open(DATASET_PATH, encoding="utf-8", newline="") as f:
    rows_in = list(csv.DictReader(f))

pipeline_rows = []
freq_raw   = collections.Counter()
freq_nosw  = collections.Counter()
freq_stemmed = collections.Counter()

for row in rows_in:
    tokens        = tokenize(row["normalized_text"])          # Step 1
    tokens_nosw   = remove_stopwords(tokens)                  # Step 2
    tokens_stemmed = stem_tokens(tokens_nosw)                 # Step 3

    freq_raw.update(tokens)
    freq_nosw.update(tokens_nosw)
    freq_stemmed.update(tokens_stemmed)

    pipeline_rows.append({
        "original_text":   row["original_text"],
        "normalized_text": row["normalized_text"],
        "tokens":          json.dumps(tokens,         ensure_ascii=False),
        "tokens_no_sw":    json.dumps(tokens_nosw,    ensure_ascii=False),
        "tokens_stemmed":  json.dumps(tokens_stemmed, ensure_ascii=False),
        "label":           row["label"],
    })

print(f"Step 1  ✅  Tokenized  {len(rows_in)} sentences")
print(f"Step 2  ✅  Stopword removal  — "
      f"{sum(freq_raw.values())} → {sum(freq_nosw.values())} tokens "
      f"({(1 - sum(freq_nosw.values())/sum(freq_raw.values()))*100:.1f}% removed)")
print(f"Step 3  ✅  Suffix stripping  — "
      f"{len(freq_nosw)} → {len(freq_stemmed)} unique types after stemming")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 4 — WRITE pipeline_dataset.csv
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
with open( "pipeline_dataset.csv", "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(pipeline_rows[0].keys()))
    w.writeheader(); w.writerows(pipeline_rows)
print(f"Step 4  ✅  pipeline_dataset.csv written")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 5 — TOKEN FREQUENCY TABLE
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
all_raw_tokens = set(freq_raw)
with open( "token_freq.csv", "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=[
        "token","count_raw","count_after_stopword_removal","count_after_stemming",
        "is_stopword","suffix_stripped","root"
    ])
    w.writeheader()
    for tok, cnt in freq_raw.most_common():
        root, sfx = strip_suffix(tok)
        w.writerow({
            "token":                          tok,
            "count_raw":                      cnt,
            "count_after_stopword_removal":   freq_nosw.get(tok, 0),
            "count_after_stemming":           freq_stemmed.get(root, 0),
            "is_stopword":                    int(tok in stopword_set),
            "suffix_stripped":                sfx or "",
            "root":                           root,
        })
print(f"Step 5  ✅  token_freq.csv written")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 6 — VOCABULARY  (built on stemmed tokens)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
word2id = {"<PAD>": 0, "<UNK>": 1}
for tok, _ in freq_stemmed.most_common():
    word2id[tok] = len(word2id)
id2word = {v: k for k, v in word2id.items()}

with open("vocab.json", "w", encoding="utf-8") as f:
    json.dump({
        "vocab_size":       len(word2id),
        "total_tokens_raw": sum(freq_raw.values()),
        "total_tokens_final": sum(freq_stemmed.values()),
        "word2id":          word2id,
        "id2word":          {str(k): v for k, v in id2word.items()},
        "stopwords":        sorted(stopword_set),
        "suffixes":         suffix_list,
    }, f, ensure_ascii=False, indent=2)
print(f"Step 6  ✅  vocab.json  — size: {len(word2id)}")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 7 — VERIFICATION REPORT
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
report_lines = []
def rpt(*args):
    line = " ".join(str(a) for a in args)
    report_lines.append(line)
    print(line)

rpt("\n" + "="*65)
rpt("VERIFICATION REPORT")
rpt("="*65)

# --- Test 1: normalization of stopwords/suffixes
rpt("\n[1] NORMALIZATION SANITY CHECKS")
pairs = [
    ("उनलाई",  "उनलाइ"),   # ई→इ, ी→ि
    ("संग",    "सग"),        # ं stripped
    ("हरू",    "हरु"),       # ू→ु
    ("बाट",    "बाट"),       # unchanged
    ("भितिर",  "भितिर"),     # त्र→तिर by norm_map
    ("को",     "कओ"),        # ो→ओ
    ("ले",     "लए"),        # े→ए
    ("मा",     "मा"),        # unchanged
]
all_ok = True
for raw, expected in pairs:
    got = normalize(raw)
    ok  = got == expected
    if not ok: all_ok = False
    rpt(f"  {'✅' if ok else '❌'}  normalize({raw!r}) = {got!r}  (expected {expected!r})")
rpt("  → All normalization checks passed ✅" if all_ok else "  → Some checks FAILED ❌")

# --- Test 2: stopword removal
rpt("\n[2] STOPWORD REMOVAL CHECKS")
sw_tests = [
    (["सास","र","कमजोरि"],        ["सास","कमजोरि"]),   # र removed
    (["उनलाइ","ज्बरओ","छ"],       ["ज्बरओ"]),           # उनलाइ, छ removed
    (["हल्का","बान्ता","छ"], ["हल्का","बान्ता"]),  # छ in sw
]
all_ok = True
for inp, expected in sw_tests:
    got = remove_stopwords(inp)
    ok  = got == expected
    if not ok: all_ok = False
    rpt(f"  {'✅' if ok else '❌'}  {inp} → {got}  (expected {expected})")
rpt("  → All stopword checks passed ✅" if all_ok else "  → Some checks FAILED ❌")

# --- Test 3: suffix stripping
rpt("\n[3] SUFFIX STRIPPING CHECKS")
sfx_tests = [
    ("सरिरबाट",   "सरिर",   "बाट"),
    ("उनलाइ",     "उन",     "लाइ"),
    ("बिहानदएखि", "बिहान",  "दएखि"),
    ("समस्यासग",  "समस्या", "सग"),
    ("छाला",      "छाला",   None),   # no suffix → unchanged
]
all_ok = True
for tok, exp_root, exp_sfx in sfx_tests:
    root, sfx = strip_suffix(tok)
    ok = root == exp_root and sfx == exp_sfx
    if not ok: all_ok = False
    rpt(f"  {'✅' if ok else '❌'}  strip({tok!r}) → root={root!r} sfx={sfx!r}  "
        f"(expected root={exp_root!r} sfx={exp_sfx!r})")
rpt("  → All suffix checks passed ✅" if all_ok else "  → Some checks FAILED ❌")

# --- Test 4: full row walkthrough
rpt("\n[4] FULL ROW WALKTHROUGH (first 3 sentences)")
for row in pipeline_rows[:3]:
    rpt(f"\n  [{row['label']}] {row['original_text'][:60]}")
    rpt(f"    normalized  : {row['normalized_text'][:70]}")
    rpt(f"    tokens      : {json.loads(row['tokens'])}")
    rpt(f"    no_stopwords: {json.loads(row['tokens_no_sw'])}")
    rpt(f"    stemmed     : {json.loads(row['tokens_stemmed'])}")

# --- Test 5: statistics
rpt("\n[5] CORPUS STATISTICS")
total_raw    = sum(freq_raw.values())
total_nosw   = sum(freq_nosw.values())
total_final  = sum(freq_stemmed.values())
rpt(f"  Sentences              : {len(rows_in)}")
rpt(f"  Tokens (raw)           : {total_raw}")
rpt(f"  Tokens (after SW rm)   : {total_nosw}  ({(1-total_nosw/total_raw)*100:.1f}% removed)")
rpt(f"  Tokens (after stemming): {total_final}")
rpt(f"  Unique types (raw)     : {len(freq_raw)}")
rpt(f"  Unique types (stemmed) : {len(freq_stemmed)}")
rpt(f"  Vocab size (w/ special): {len(word2id)}")
rpt(f"  Stopwords used         : {len(stopword_set)}")
rpt(f"  Suffix rules used      : {len(suffix_list)}")

rpt("\n" + "="*65)
rpt("OUTPUT FILES")
rpt("="*65)
from pathlib import Path

for fname in ["normalized_stopwords.json","normalized_suffixes.json",
              "pipeline_dataset.csv","token_freq.csv","vocab.json"]:
    p = Path(fname)
    rpt(f"  {fname:35s}  {p.stat().st_size:,} bytes")

# Save report
with open( "pipeline_report.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines))
print("\n✅  pipeline_report.txt saved")

Step 0  ✅  Stopwords : 219 raw → 194 normalized unique
Step 0  ✅  Suffixes  : 228 raw → 112 normalized unique
Step 1  ✅  Tokenized  705 sentences
Step 2  ✅  Stopword removal  — 7209 → 5745 tokens (20.3% removed)
Step 3  ✅  Suffix stripping  — 1535 → 1350 unique types after stemming
Step 4  ✅  pipeline_dataset.csv written
Step 5  ✅  token_freq.csv written
Step 6  ✅  vocab.json  — size: 1352

VERIFICATION REPORT

[1] NORMALIZATION SANITY CHECKS
  ✅  normalize('उनलाई') = 'उनलाइ'  (expected 'उनलाइ')
  ✅  normalize('संग') = 'सग'  (expected 'सग')
  ✅  normalize('हरू') = 'हरु'  (expected 'हरु')
  ✅  normalize('बाट') = 'बाट'  (expected 'बाट')
  ✅  normalize('भितिर') = 'भितिर'  (expected 'भितिर')
  ✅  normalize('को') = 'कओ'  (expected 'कओ')
  ✅  normalize('ले') = 'लए'  (expected 'लए')
  ✅  normalize('मा') = 'मा'  (expected 'मा')
  → All normalization checks passed ✅

[2] STOPWORD REMOVAL CHECKS
  ✅  ['सास', 'र', 'कमजोरि'] → ['सास', 'कमजोरि']  (expected ['सास', 'कमजोरि'])
  ✅  ['उनलाइ', 'ज्बरओ',